# ViT tier: does retrieval training add anything?

The trainable half of the ViT tier, read from `results/runs.jsonl`. Every
trained row is a checkpoint `cbir train` produced here and `cbir evaluate vit
--checkpoint` scored through the same path as a frozen model; its "before" is the same
network frozen, under the same geometry.

That geometry is `stretch` — IRT's protocol, every image squashed into the square — not
the `pad` of `05_vit_comparison.ipynb`. Training runs in it, so before and after
are both measured in it.

Two questions:

1. **Does the recipe reproduce?** IRT (El-Nouby et al. 2021) on DeiT, against the paper's
   Table 7.
2. **Does it add anything to DINO?** The same recipe, full fine-tuning and LoRA, against
   the frozen DINO row.

All mAP on 0-1. Per-epoch training curves are not plotted: these runs predate the fix that
made a broken trackio install warn instead of passing for a missing one, so none were
recorded. Each checkpoint still carries its best epoch and that epoch's validation mAP.

In [ ]:
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

sys.path.insert(0, ".")  # style.py sits beside this notebook
import style
from cbir.eval.frame import tidy_rows
from cbir.eval.results import latest

style.use()
DOCS = Path("..") / "docs"
RUNS = Path("..") / "data" / "runs"
DOCS.mkdir(exist_ok=True)

## The data

Trained rows carry a `checkpoint`; its directory names the run, and the run name encodes
what was trained (`-lora8` for LoRA, `-lr…` for the learning rate). The PCA-whitened
evaluation of one checkpoint is left out of the charts: whitening lowered it, so the
rows are raw CLS, as the paper reports them.

In [ ]:
df = pd.DataFrame(tidy_rows(records=list(latest().values())))
vit = df[df["technique"] == "vit"].copy()
for key, default in {"resize": "pad", "readout": "native", "whiten": False, "checkpoint": None}.items():
    vit[key] = [p.get(key, default) for p in vit["params"]]
vit["model"] = [p["model"] for p in vit["params"]]
vit["image_size"] = [p["image_size"] for p in vit["params"]]
vit = vit[(vit["resize"] == "stretch") & (vit["readout"] == "native") & ~vit["whiten"].astype(bool)]

NAME = {"deit-vits16-dist-224": "DeiT S/16† 224", "deit-vitb16-dist-224": "DeiT B/16† 224",
        "dinov2-vits14": "DINOv2 S/14", "dinov2-vitb14": "DINOv2 B/14", "dinov3-vitl16": "DINOv3 L/16"}

def how(run):
    lora = re.search(r"-lora(\d+)", run)
    lr = re.search(r"-lr(.+?)-(?:lora|seed)", run).group(1)
    if lora:
        # `1e-05` -> `1e-5`: the run name keeps Python's float repr, the label need not.
        mantissa, exponent = f"{float(lr):.0e}".split("e")
        return f"LoRA r{lora.group(1)}, lr {mantissa}e{int(exponent)}"
    return "IRT recipe" if run.startswith("deit") else "full fine-tuning"

trained = vit[vit["checkpoint"].notna()].copy()
trained["run"] = trained["checkpoint"].map(lambda path: Path(path).parent.name)
trained["config"] = [f"{NAME[m]} · {how(r)}" for m, r in zip(trained["model"], trained["run"])]
trained["lr"] = trained["run"].map(lambda run: float(re.search(r"-lr(.+?)-(?:lora|seed)", run).group(1)))
trained["lora"] = trained["run"].str.contains("-lora")
frozen = vit[vit["checkpoint"].isna()]

def before(model, image_size, dataset, protocol):
    rows = frozen[(frozen["model"] == model) & (frozen["image_size"] == image_size)
                  & (frozen["dataset"] == dataset) & (frozen["protocol"] == protocol)]
    return rows["map"].max()

trained["frozen_map"] = [before(m, s, d, p) for m, s, d, p in
                         zip(trained["model"], trained["image_size"], trained["dataset"], trained["protocol"])]
trained["change"] = trained["map"] - trained["frozen_map"]
trained[trained["protocol"] != "easy"].pivot_table(
    index="config", columns=["dataset", "protocol"], values="change").round(3)

## Before and after, every trained configuration

Blue is the network frozen, orange the same network after training; the grey bar between
them is what training moved. The black ticks on the DeiT rows are IRT's Table 7 — the
published result of the same recipe, the target for reproduction.

In [ ]:
IRT_TABLE7 = {  # (model, dataset, protocol) -> published mAP, on 0-1
    ("deit-vits16-dist-224", "roxford5k", "medium"): 0.345, ("deit-vits16-dist-224", "roxford5k", "hard"): 0.158,
    ("deit-vits16-dist-224", "rparis6k", "medium"): 0.658, ("deit-vits16-dist-224", "rparis6k", "hard"): 0.420,
    ("deit-vitb16-dist-224", "roxford5k", "medium"): 0.395, ("deit-vitb16-dist-224", "roxford5k", "hard"): 0.174,
    ("deit-vitb16-dist-224", "rparis6k", "medium"): 0.675, ("deit-vitb16-dist-224", "rparis6k", "hard"): 0.436,
}

# Top to bottom: DeiT, then DINO by size; within a model full fine-tuning, then LoRA by lr.
ranked = trained.assign(rank=trained["model"].map(list(NAME).index))
ORDER = ranked.sort_values(["rank", "lora", "lr"])["config"].drop_duplicates().tolist()[::-1]

fig_training, axes = plt.subplots(2, 2, figsize=(11.5, 7.6), sharey=True)
for row, dataset in enumerate(["roxford5k", "rparis6k"]):
    for col, protocol in enumerate(["medium", "hard"]):
        ax = axes[row, col]
        rows = trained[(trained["dataset"] == dataset) & (trained["protocol"] == protocol)].set_index("config")
        for i, config in enumerate(ORDER):
            if config not in rows.index:
                continue
            r = rows.loc[config]
            ax.plot([r["frozen_map"], r["map"]], [i, i], color=style.GRID, linewidth=5, solid_capstyle="round", zorder=2)
            ax.scatter(r["frozen_map"], i, s=60, color=style.STATE["frozen"], edgecolor=style.SURFACE, linewidth=1.5, zorder=3)
            ax.scatter(r["map"], i, s=60, color=style.STATE["trained here"], edgecolor=style.SURFACE, linewidth=1.5, zorder=4)
            style.label_point(ax, max(r["map"], r["frozen_map"]), i, f"{r['change']:+.3f}", dx=9)
            published = IRT_TABLE7.get((r["model"], dataset, protocol))
            if published is not None:
                ax.plot([published, published], [i - 0.32, i + 0.32], color=style.INK, linewidth=2, zorder=5)
        ax.set_yticks(range(len(ORDER)), ORDER, fontsize=8.5)
        ax.set_xlim(0, 1.0)
        style.style(ax, title=f"{dataset} {protocol.capitalize()}", xlabel="mAP" if row else None, axis="x")

handles = [plt.Line2D([], [], marker="o", linestyle="", markersize=8, color=c) for c in style.STATE.values()]
handles.append(plt.Line2D([], [], marker="|", linestyle="", markersize=12, markeredgewidth=2, color=style.INK))
fig_training.legend(handles, [*style.STATE, "IRT Table 7"], loc="upper center", ncol=3, frameon=False,
                    bbox_to_anchor=(0.5, 1.0))
fig_training.tight_layout(rect=(0, 0, 1, 0.95))  # the top band is the legend's

- **IRT reproduces.** Both DeiT rows land on the published tick or just past it — S†
  within 0.008 everywhere, B† within 0.006 on rparis6k and about 0.03 above on roxford5k,
  one seed each side. Training lifts DeiT by 0.08-0.13 on roxford5k and 0.03-0.09 on
  rparis6k.
- **On DINO every bar points left or nowhere.** Full fine-tuning lowers DINOv2 S and B,
  no LoRA learning rate beats frozen DINOv2 S, and LoRA leaves DINOv3 L/16 within noise of
  its frozen score. The one cell that rises on DINOv2 S (LoRA lr 1e-4, roxford5k Hard) has
  its Medium below frozen.

## Why validation could not have warned us

Each run keeps the checkpoint with the best mAP on the SfM validation split — the
landmarks the recipe trains on, held out. On DeiT that number and the benchmark move
together. On DINO they come apart: every run climbs on SfM and every run loses on
rparis6k, and across the three LoRA learning rates the higher climb is the bigger loss.
Full fine-tuning climbs highest of all without being the worst, so this is an ordering
within one method, not a law across methods — and it is drawn that way: the LoRA runs as
one line in learning-rate order, full fine-tuning as a separate point.

The SfM numbers come from each checkpoint's own metadata in `data/runs/` (local, not
committed), and the frozen model's from `scripts/sfm_validation.py`.

In [ ]:
FROZEN_SFM = {"dinov2-vits14": 0.8575}  # scripts/sfm_validation.py, the same metric as epoch 0

def best_sfm(checkpoint):
    path = Path("..") / checkpoint
    if not path.exists():
        return None
    return torch.load(path, map_location="cpu", weights_only=False)["meta"]["metrics"]["mean_average_precision"]

subject = trained[(trained["model"] == "dinov2-vits14") & (trained["dataset"] == "rparis6k")].copy()
subject["sfm"] = subject["checkpoint"].map(best_sfm)

if subject["sfm"].isna().any():
    print("checkpoint metadata not found under", RUNS.resolve(), "- skipping")
else:
    fig_sfm, axes = plt.subplots(1, 2, figsize=(10.5, 3.9), sharey=True)
    for ax, protocol in zip(axes, ["medium", "hard"]):
        rows = subject[subject["protocol"] == protocol].sort_values("sfm")
        ax.axhline(0, color=style.MUTED, linewidth=1.1, linestyle=(0, (4, 3)), zorder=2)
        style.label_point(ax, FROZEN_SFM["dinov2-vits14"], 0, "frozen", dx=4, dy=6, va="bottom")
        ax.scatter([FROZEN_SFM["dinov2-vits14"]], [0], s=60, color=style.STATE["frozen"],
                   edgecolor=style.SURFACE, linewidth=1.5, zorder=3)
        lora = rows[rows["lora"]].sort_values("lr")
        ax.plot(lora["sfm"], lora["change"], color=style.STATE["trained here"], linewidth=2, zorder=3,
                marker="o", markersize=8, markeredgecolor=style.SURFACE, markeredgewidth=1.5)
        full = rows[~rows["lora"]]
        ax.scatter(full["sfm"], full["change"], s=70, marker="D", color=style.STATE["trained here"],
                   edgecolor=style.SURFACE, linewidth=1.5, zorder=4)
        for _, r in rows.iterrows():
            label = r["config"].split(" · ")[1].replace("LoRA r8, ", "LoRA ")
            style.label_point(ax, r["sfm"], r["change"], label, dx=8, dy=0, ha="left", fontsize=7.5)
        ax.set_xlim(0.855, 0.897)
        style.style(ax, title=f"DINOv2 S/14, rparis6k {protocol.capitalize()}",
                    xlabel="best SfM validation mAP", ylabel="change vs frozen" if protocol == "medium" else None)
    fig_sfm.tight_layout()

Every trained point sits right of frozen on SfM and below it on rparis6k. Selecting a
learning rate on SfM validation — the only selection the protocol allows, since tuning
on Oxford or Paris is ruled out — would pick lr 1e-4, the worst of the LoRA rows here.

**So the trainable half closes with a negative result:** on this recipe and this data,
retrieval training adds nothing to DINO. Self-supervised pretraining already sits past
where SfM-120k can take a network, and moving further toward SfM moves away from Oxford
and Paris. Reopening it needs a different training signal, not another schedule.

## Write the docs figures

In [ ]:
figures = [("vit_training", fig_training)]
if "fig_sfm" in globals():
    figures.append(("vit_sfm_vs_benchmark", fig_sfm))
# White rather than transparent, for GitHub's dark theme.
for name, figure in figures:
    path = DOCS / f"{name}.png"
    figure.savefig(path, dpi=110, bbox_inches="tight", facecolor="white")
    print("wrote", path.resolve())